# RAG Implementation

To run this notebook successfully, a Groq API key and Ollama running locally is required. 

Groq:
- Put the Groq API key as `GROQ_API_KEY=...` in the `.env` file

Ollama:
- Install Ollama and run `ollama serve`
- Install the embedding model with `ollama pull nomic-embed-text`

In [1]:
# Imports
import pandas as pd
import numpy as np
import requests
import csv
from groq import Groq
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown

In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [3]:
# Ollama endpoint
OLLAMA = "http://localhost:11434"

# Models
EMBED_MODEL = "nomic-embed-text"
CHAT_MODEL = "openai/gpt-oss-20b"

## Embedding and LLM Setup

First we need to embed the documents we want to use for RAG. Then, we can send the embeddings along with our query to the LLM.

In [4]:
# Function to send requests to embed model
def embed(texts):
    r = requests.post(f"{OLLAMA}/api/embed", json={"model": EMBED_MODEL, "input": texts})
    return np.array(r.json()["embeddings"])

In [5]:
# Function to send requests to LLM model
def ask_llm(prompt):
    client = Groq()

    try:
        r = client.chat.completions.create(
            model=CHAT_MODEL,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        return r.choices[0].message.content
    except Exception as e:
        print(f"Error: {e}")

In [6]:
ask_llm("What is the capital of France?")

'The capital of France is **Paris**.'

## Process and Load in Data

Now we'll want to load in the recall data and perform some data cleaning. Load the data into a DataFrame first, then convert the DataFrame rows into a formatted text document.

In [7]:
# Load data in a DataFrame
DATA_PATH = Path("../data/raw/FLAT_RCL_PRE_2010.txt") # notebook lives in notebooks/

COLUMNS = [
    "RECORD_ID", "CAMPNO", "MAKETXT", "MODELTXT", "YEARTXT", "MFGCAMPNO",
    "COMPNAME", "MFGNAME", "BGMAN", "ENDMAN", "RCLTYPECD", "POTAFF", "ODATE",
    "INFLUENCED_BY", "MFGTXT", "RCDATE", "DATEA", "RPNO", "FMVSS",
    "DESC_DEFECT", "CONEQUENCE_DEFECT", "CORRECTIVE_ACTION", "NOTES",
    "RCL_CMPT_ID", "MFR_COMP_NAME", "MFR_COMP_DESC", "MFR_COMP_PTNO",
    "DO_NOT_DRIVE", "PARK_OUTSIDE"
]

df = pd.read_csv(
    DATA_PATH,
    sep="\t", # Data is stored as .txt but is separated by tabs
    header=None,           
    names=COLUMNS,
    dtype=str,             
    encoding="latin-1",
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
)

In [8]:
df.head()

,RECORD_ID,CAMPNO,MAKETXT,MODELTXT,YEARTXT,MFGCAMPNO,COMPNAME,MFGNAME,BGMAN,ENDMAN,...,DESC_DEFECT,CONEQUENCE_DEFECT,CORRECTIVE_ACTION,NOTES,RCL_CMPT_ID,MFR_COMP_NAME,MFR_COMP_DESC,MFR_COMP_PTNO,DO_NOT_DRIVE,PARK_OUTSIDE
0,1,66V026000,FORD,BRONCO,1967,,STEERING:LINKAGES:DRAG:LINK:CONNECTION,FORD MOTOR COMPANY,,,...,,,,POSSIBILITY THAT INCORRECT STEERING DRAG LINK ...,000000042000027961000000016,,,,No,No
1,2,66V011001,GMC,SUBURBAN,1966,,SEAT BELTS:FRONT:ANCHORAGE,GENERAL MOTORS CORP.,,,...,,,,POSSIBLE INTERFERENCE BETWEEN LEFT REAR TIRE A...,000000022000045867000000265,,,,No,No
2,3,66V008002,DODGE,DART,1967,,LATCHES/LOCKS/LINKAGES:DOORS:LATCH,CHRYSLER CORPORATION,,,...,,,,DOOR LATCH STRIKER ASSEMBLIES MAY HAVE A BRITT...,000000017000029125000000289,,,,No,No
3,4,66V008002,PLYMOUTH,VALIANT,1967,,LATCHES/LOCKS/LINKAGES:DOORS:LATCH,CHRYSLER CORPORATION,,,...,,,,DOOR LATCH STRIKER ASSEMBLIES MAY HAVE A BRITT...,000000017000045859000000289,,,,No,No
4,5,66V008001,DODGE,DART,1966,,WHEELS,CHRYSLER CORPORATION,,,...,,,,POSSIBLE INTERFERENCE BETWEEN THE FLANGE OF TH...,000000016000045857000000309,,,,No,No


In [9]:
# Build chunks from rows of the DataFrame
def build_doc(g):
    first = g.iloc[0]
    models = sorted(set(g["MAKETXT"] + " " + g["MODELTXT"] + " " + g["YEARTXT"]))
    text = (
        f"Recall campaign {first['CAMPNO']}.\n"
        f"Manufacturer: {first['MFGNAME']}.\n"
        f"Component: {first['COMPNAME']}.\n"
        f"Affected vehicles: {'; '.join(models)}.\n"
        f"Defect: {first['DESC_DEFECT']}\n"
        f"Consequence: {first['CONEQUENCE_DEFECT']}\n"
        f"Remedy: {first['CORRECTIVE_ACTION']}"
    )
    return {
        "id": first["CAMPNO"],
        "text": text,
        "makes": sorted(set(g["MAKETXT"])),
        "years": sorted(set(g["YEARTXT"])),
        "component": first["COMPNAME"],
    }

docs = [build_doc(g) for _, g in df.groupby("CAMPNO")]

In [10]:
# Embed each chunk's `text` (turn chunks into vectors)
embeddings_path = Path("../data/processed/embeddings.npy")
texts = ["search_document: " + doc["text"] for doc in docs]

# Skip redoing embeddings if embeddings data already exists
if embeddings_path.is_file():
    chunk_vectors = np.load(embeddings_path)
else:
    chunk_vectors = np.vstack([embed(texts[i:i+64]) for i in range(0, len(texts), 64)])
    np.save("../data/processed/embeddings.npy", chunk_vectors) # Avoid re-embedding on every restart

    print(f"Indexed {len(docs)} chunks")

## Searching / Retrieval
With the chunk vectors, we now want to search through them based on similarity to a query. Take the user's question and turn it into a vector as well, then compare it to each chunk with cosine similarity. Return the top *k* most similar chunks to the user query.

In [11]:
# Function to retrieve the top k most similar chunks to a query
def retrieve(query, k=5):
    q = embed([query])[0]
    scores = chunk_vectors @ q / (np.linalg.norm(chunk_vectors, axis=1) * np.linalg.norm(q))
    return [texts[i] for i in np.argsort(scores)[::-1][:k]]

## Calling the LLM with RAG

Attach the retrieved chunks as context to the user's query and send it to the LLM.

In [12]:
def ask_llm_rag(question):
    context = "\n\n".join(retrieve(question))
    prompt = f"""Answer the question using ONLY the context below.
If the answer isn't in the context, say "I don't know."

Context:
{context}

Question: {question}"""

    return ask_llm(prompt)

In [13]:
question = "What is the recall for the 2005 Ford F-150?"

response = ask_llm_rag(question)
display(Markdown(response))

Recall campaign 06V033000 – certain 2005 Ford F‑150 trucks were built with a software error in the instrument cluster that could keep the ABS warning light from illuminating if the ABS module failed. Dealers will recalibrate the instrument cluster. The recall began February 27, 2006 and owners may call Ford at 1‑800‑392‑3673.

In [14]:
# Chat loop (q to quit)
while True:
    question = input("Ask a question: ")

    if question.lower() == "q":
        break
    
    display(Markdown(f"**Question:** {question}"))

    response = ask_llm_rag(question)
    display(Markdown(response))

**Question:** Has Tesla made any recalls?

Yes. Tesla has issued a recall for 345 2008 Tesla Roadster vehicles.

**Question:** Has the Toyota Camry ever been recalled?

Yes, the Toyota Camry has been recalled.

**Question:** What are the Honda Civic recalls?

**Honda Civic recall campaigns (as listed in the supplied context)**  

| Recall ID | Year/Model(s) affected | Component | Key defect / issue | Potential consequence | Remedy (what dealers will do) |
|-----------|------------------------|-----------|---------------------|-----------------------|--------------------------------|
| **99I002000** | 1996 Honda Civic | Vehicle Speed Control | Accelerator pedal feels rough/notchy; may feel unresponsive at low speeds | Not a safety recall but a safety‑improvement campaign | Dealers will **replace the throttle cable** |
| **06V326000** | 2006 Honda Civic (2‑door) | Visibility: glass, side/rear | Rear windshield or rear quarter‑glass panels do not adhere properly due to improper window‑opening preparation | Panels can become loose, rattle, lift or separate, increasing crash risk | Dealers will **remove the glass, prepare the window opening, and reinstall the glass** |
| **07V034000** | 2006 Honda Civic Hybrid | Engine & engine cooling – Integrated Motor Assist (IMA) | Metal cover behind the rear seat can pinch the rubber cap on an electrical terminal, possibly causing a short | Short may blow the IMA fuse → engine stalls → higher crash risk | Dealers will **install a plastic cover attached to a metal bracket over the rubber cap** |
| **08V535000** | 2008‑2009 Honda Civic (sedan) | Fuel system – hose, line, piping, fittings | Bracket covering the fuel hose connector was not secured with a nut | In a crash the bracket could loosen, damaging the connector → fuel leak → fire risk | Dealers will **install a nut on the fuel‑feed‑hose bracket** |
| **73V106000** | 1973 Honda Civic | Exterior lighting – hazard flashing warning light switch | No defect, consequence, or remedy information provided in the context |  |  |

These are the Honda Civic recall campaigns recorded in the supplied material.

**Question:** How many times has the Toyota Prius been recalled?

I don't know.

**Question:** What is the capital of France?

I don't know.